# Prompting Techniques with Ollama

Prompting steers a pretrained LLM at inference time, with **no weight updates**. In this notebook you will:

1. See four techniques in action: **few-shot**, **chain-of-thought**, **self-consistency**, **RAG**
2. Practice them yourself in short exercises

| Technique | Idea |
|---|---|
| Zero/Few-shot | Give instructions, optionally with examples |
| Chain-of-Thought (CoT) | Ask for step-by-step reasoning before the answer |
| Self-Consistency | Sample several CoT chains, majority-vote the answer |
| RAG | Retrieve relevant documents and put them in the prompt |

## Setup
Run once. Requires **Ollama** running locally (`ollama serve`) and the `qwen3:14b` model pulled.

In [1]:
import ollama
models = [m['model'] for m in ollama.list().models]
assert 'qwen3:14b' in models, f'qwen3:14b not found. Run: ollama pull qwen3:14b'
print('Ollama is running. Models:', models)

Ollama is running. Models: ['qwen3.8-32k:latest', 'qwen3.8-coder:latest', 'qwen3.8:27b', 'my-model-backup:latest', 'orcarouter/Qwen3.8-27B-Uncensored:q4_K_M', 'deepseek-r1:32b', 'qwen3-coder:30b', 'qwen3:14b']


In [2]:
import ollama, re
from collections import Counter

MODEL = "qwen3:14b"

def generate(prompt, max_tokens=150, temperature=0.0):
    """Send a prompt to the model and return its text. temperature=0 is deterministic."""
    r = ollama.generate(model=MODEL, prompt=prompt, think=False,
                        options={"temperature": temperature, "num_predict": max_tokens})
    return r["response"].strip()

print(generate("Say hello in one short sentence."))

Hello!


## Demo 1: Zero-shot vs. Few-shot
Few-shot adds input→output examples so the model copies the format.

In [3]:
review = "The pacing was dreadful, but the soundtrack was gorgeous."
task = "Classify the review as Positive, Negative, or Mixed. Reply with one word."

zero_shot = f"{task}\n\nReview: {review}\nLabel:"

few_shot = f"""{task}

Review: A masterpiece of modern cinema.
Label: Positive

Review: I fell asleep halfway through.
Label: Negative

Review: Great acting, but the plot made no sense.
Label: Mixed

Review: {review}
Label:"""

print("Zero-shot:", generate(zero_shot, max_tokens=10))
print("Few-shot :", generate(few_shot, max_tokens=10))

Zero-shot: Mixed


Few-shot : Mixed


## Demo 2: Chain-of-Thought
Asking the model to reason step by step helps on multi-step problems.

In [4]:
question = ("A store had 50 shirts. It sold 18 on Monday, got a shipment of 30, "
            "then sold half of what it had. How many shirts are left?")   # answer: 31

print("Direct:\n", generate(f"Q: {question}\nGive only the final number.\nA:", max_tokens=10))
print("\nZero-shot CoT:\n", generate(f"Q: {question}\nA: Let's think step by step.", max_tokens=200))

Direct:
 17



Zero-shot CoT:
 Let's solve the problem step by step:

1. **Start with the initial number of shirts:**
   - The store had **50 shirts**.

2. **Shirts sold on Monday:**
   - It sold **18 shirts**.
   - Remaining shirts:  
     $ 50 - 18 = 32 $ shirts.

3. **Received a shipment of 30 shirts:**
   - Now the store has:  
     $ 32 + 30 = 62 $ shirts.

4. **Sold half of what it had:**
   - It sold:  
     $ \frac{62}{2} = 31 $ shirts.
   - Remaining shirts:  
     $ 62 - 31 = 31 $ shirts.

### ✅ Final Answer: **31 shirts** are left.


## Demo 3: Self-Consistency
Sample several reasoning chains (temperature > 0) and take the most common final answer.

In [5]:
def final_number(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else None

prompt = (f"Q: {question}\nThink step by step, then end with 'The answer is <number>.'\nA:")
answers = []
for _ in range(5):
    out = generate(prompt, max_tokens=250, temperature=0.8)
    answers.append(final_number(out))

print("Votes :", Counter(answers))
print("Winner:", Counter(answers).most_common(1)[0][0], "(truth: 31)")

Votes : Counter({'31': 5})
Winner: 31 (truth: 31)


## Demo 4: RAG
Retrieve the most relevant document (TF-IDF here; real systems use embeddings + a vector DB) and add it to the prompt.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = [
    "Acme Corp's refund policy: items can be returned within 45 days with a receipt.",
    "Acme Corp support hours are 9am-5pm ICT, Monday to Friday.",
    "Acme Corp's Pro plan costs $29 per month and includes 5 seats.",
    "Acme Corp headquarters are located in Phnom Penh, Cambodia.",
]
vec = TfidfVectorizer().fit(docs)
doc_vecs = vec.transform(docs)

def retrieve(query, k=1):
    scores = cosine_similarity(vec.transform([query]), doc_vecs).ravel()
    return [docs[i] for i in scores.argsort()[::-1][:k]]

def rag_answer(query, k=1):
    context = "\n".join(retrieve(query, k))
    return generate(f"Answer using only the context.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:")

q = "How many days do I have to return an item at Acme Corp?"
print("No context:", generate(f"Question: {q}\nAnswer:", max_tokens=60))
print("With RAG  :", rag_answer(q))

No context: Acme Corp's return policy may vary depending on the specific product and location. However, a typical return policy for many retailers is a 30-day return window from the date of purchase.

For the most accurate and up-to-date information, you should check Acme Corp's official website or contact


With RAG  : The context provided does not mention anything about the return period for items at Acme Corp. Therefore, it is not possible to determine how many days you have to return an item based on the given information.


---
# Hands-on Practice
Complete the `TODO`s. Re-run each cell and compare outputs. Small models are noisy, so try a few variations.

### Exercise 1: Few-shot extraction
Make the model turn a sentence into JSON with keys `name`, `city`, `age`. First try zero-shot, then add **2 examples** to get a clean, consistent format.

In [7]:
text = "Sokha, a 29-year-old engineer, just moved to Siem Reap."

prompt = f"""Extract name, city, and age as JSON.

Text: Dara, a 25-year-old teacher, just moved to Phnom Penh.
JSON: {{"name": "Dara", "city": "Phnom Penh", "age": 25}}

Text: Vanna, a 34-year-old chef, just moved to Battambang.
JSON: {{"name": "Vanna", "city": "Battambang", "age": 34}}

Text: {text}
JSON:"""

print(generate(prompt, max_tokens=60))

JSON: {"name": "Sokha", "city": "Siem Reap", "age": 29}


### Exercise 2: Chain-of-Thought
The puzzle below often trips up direct answers. Compare a **direct** prompt with a **CoT** prompt. Which is right? (Answer: 0.05, i.e. 5 cents)

In [8]:
puzzle = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much is the ball?"

direct = f"Q: {puzzle}\nGive only the final answer.\nA:"
cot    = f"Q: {puzzle}\nA: Let's think step by step."

print("Direct:", generate(direct, max_tokens=40))
print("CoT   :", generate(cot, max_tokens=200))

Direct: A: $0.05


CoT   : Let's solve this step by step.

### Step 1: Understand the problem
We are told:

- A **bat and a ball together cost $1.10**.
- The **bat costs $1.00 more than the ball**.

We are asked: **How much is the ball?**

---

### Step 2: Assign variables
Let’s let:

- $ x $ = the cost of the **ball** (in dollars)
- Then the **bat** costs $ x + 1.00 $ (since it's $1.00 more than the ball)

---

### Step 3: Set up the equation
The total cost is $1.10:

$$
x + (x + 1.00) = 1.10
$$

---

### Step 4: Solve the equation

$$
x + x + 1.00 = 1.10
$$
$$
2


### Exercise 3: Self-Consistency
Write a function `self_consistent(prompt, k)` that samples `k` answers at `temperature=0.8` and returns the majority answer. Test it on the puzzle above with `k=1` vs `k=7`. Does voting help?

*Hint:* ask the model to end with `Answer: <number>` so extracting the answer is easy.

In [9]:
def self_consistent(prompt, k=5):
    answers = []
    for _ in range(k):
        out = generate(prompt, max_tokens=250, temperature=0.8)
        m = re.search(r"Answer:\s*([\d.]+)", out)
        answers.append(m.group(1) if m else out.strip())
    return Counter(answers).most_common(1)[0][0]

sc_prompt = f"Q: {puzzle}\nThink step by step, then end with 'Answer: <number>'.\nA:"
print("k=1:", self_consistent(sc_prompt, k=1))
print("k=7:", self_consistent(sc_prompt, k=7))

k=1: 0.05


k=7: 0.05


### Exercise 4: RAG
1. Add **two new facts** of your own to `docs` (re-fit the vectorizer after).
2. Ask a question that only your facts can answer.
3. Compare the answer with and without retrieval. What happens when the answer is **not** in the docs? Edit the RAG prompt so the model says *"I don't know"* in that case.

In [10]:
# Add two new facts
docs.append("Acme Corp's mobile app is available on both iOS and Android.")
docs.append("Acme Corp was founded in 2015 by three engineers in Phnom Penh.")

# Re-fit the vectorizer
vec = TfidfVectorizer().fit(docs)
doc_vecs = vec.transform(docs)

# Updated RAG prompt with 'I don't know' fallback
def rag_answer(query, k=1):
    context = "\n".join(retrieve(query, k))
    return generate(f"Answer using only the context below. If the answer is not in the context, say 'I don't know'.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:")

# Test: question only the new facts can answer
q = "When was Acme Corp founded?"
print("No context:", generate(f"Question: {q}\nAnswer:", max_tokens=60))
print("With RAG  :", rag_answer(q))

# Test: question NOT in the docs
q2 = "What is the name of Acme Corp's CEO?"
print("\nNot in docs:")
print("With RAG  :", rag_answer(q2))

No context: Acme Corp is a fictional company often used in comedic contexts, particularly in the works of cartoonist and animator Chuck Jones, most famously in the "Looney Tunes" and "Merrie Melodies" cartoons. Since it is a fictional entity, Acme Corp does not have a real founding


With RAG  : Acme Corp was founded in 2015.

Not in docs:


With RAG  : I don't know.


---
## Submission
Push your completed notebook to your GitHub repo:

```bash
git add .
git commit -m "Session 2: Prompting Techniques with Ollama"
git remote add origin https://github.com/Chumdararith-LOU/llm-fundamentals.git
git branch -M main
git push -u origin main
```

Attach your **submission.txt** (see below) to the push.

In [11]:
# Generate submission.txt with your name and key findings
submission = """LLM Fundamentals - Session 2 Submission
=========================================
Name: Chumdararith LOU
Date: 2026-10-29
Model: qwen3:14b (local Ollama)

Exercise 1 - Few-shot extraction:
  Zero-shot output: (paste your output)
  Few-shot output: (paste your output)
  Did examples help format consistency? (yes/no + note)

Exercise 2 - Chain-of-Thought:
  Direct answer: (paste)
  CoT answer:    (paste)
  Which was correct? (Direct/CoT)

Exercise 3 - Self-Consistency:
  k=1 answer: (paste)
  k=7 answer: (paste)
  Did voting help? (yes/no + note)

Exercise 4 - RAG:
  New facts added: (list them)
  Question asked:  (paste)
  No-context answer: (paste)
  RAG answer:        (paste)
  Not-in-docs answer: (paste - should say 'I don't know')
"""
with open("submission.txt", "w") as f:
    f.write(submission)
print("submission.txt created - fill in your outputs and commit.")

submission.txt created - fill in your outputs and commit.


### Challenge (optional)
Pick a task from your own work (e.g. classify support tickets, summarize, translate to Khmer). Design a prompt, then improve it by adding **a role, a format constraint, and examples**. Note which change helped most.

---
## Recap
- **Few-shot**: examples fix format and style
- **CoT**: reasoning steps help multi-step problems
- **Self-consistency**: voting across samples improves reliability, at k× the cost
- **RAG**: grounds answers in your own data

Next: when prompting is not enough, **fine-tuning** updates the weights.